In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_config
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler # used to normalize data for radar charts # conda install -c conda-forge scikit-learn   
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw # pip install fastdtw

# To handle curvature issue 
# robot stagnate or slowly moves curvature explode
from scipy.signal import medfilt
from scipy.interpolate import interp1d

In [ ]:
def compute_node_errors(all_data, reference_nodes):
    """
    all_data: dataframe containing pose_x, pose_y, closest_node, robot, environment, augmentation
    reference_nodes: dictionary mapping (robot, env) -> reference dataframe with node_idx, node_x_odom, node_y_odom
    """
    results = []

    # Iterate per (robot, environment, augmentation)
    for (robot, env, _), df_group in all_data.groupby(["robot", "environment", "augmentation"]):
        ref_key = (robot, env)
        if ref_key not in reference_nodes:
            print(f"No reference nodes for ({robot}, {env})")
            continue

        ref_df = reference_nodes[ref_key]

        ref_positions = ref_df[["node_x_odom", "node_y_odom"]].to_numpy()
        poses = df_group[["pose_x", "pose_y"]].to_numpy()

        # Compute pairwise distances
        distances = np.linalg.norm(poses[:, None, :] - ref_positions[None, :, :], axis=2)  # shape (n_poses, n_nodes)

        # Find index of closest node
        closest_ref_idx = np.argmin(distances, axis=1)
        true_nodes = ref_df.iloc[closest_ref_idx]["node_idx"].values # the dataframe index of the closest node is node_idx hint look at dataframes generated in closest_node_analysis

        # Compute node prediction error
        df_group = df_group.copy()
        df_group["true_node"] = true_nodes
        df_group["node_error"] = df_group["closest_node"] - df_group["true_node"]

        results.append(df_group)

    # Merge all groups back together
    return pd.concat(results, ignore_index=True)



**Handling velocity**

In [ ]:
# TODO handle config
import yaml

CONFIG_PATH = "../../src/visualnav-transformer/deployment/config/robot.yaml"
with open(CONFIG_PATH, "r") as f:
	robot_config = yaml.safe_load(f)
DT = 1/robot_config["frame_rate"]

In [ ]:
def model_only_segments(df, teleop_col='lin_x_teleop'):
    """Return boolean mask and segment indices for contiguous model-controlled segments."""
    mask_model_only = df[teleop_col] == 0.0
    # label contiguous True regions
    # diff on mask: True->False transitions produce boundaries
    idx = np.where(mask_model_only)[0]
    if len(idx) == 0:
        print("No model segments found")
        return []  # no model segments
    # find breaks where indices are not consecutive
    breaks = np.where(np.diff(idx) != 1)[0] # if difference between consecutive indices is not 1, we have a jump in indices meaning we used teleoperation
    segments = []
    start = idx[0]
    for b in breaks:
        end = idx[b]
        segments.append((start, end))
        start = idx[b+1]
    segments.append((start, idx[-1]))
    return segments

# polar plot


### handle multiple dfs

### single dfs

## Process data

In [ ]:
reference_header = 'reference'

config = load_config()
# TODO Handle now that we cannot be inside docker using plotly anymore

df = pd.read_csv(f"../dataframes/all_data_20251014-180242.csv") #Index(['pose_x', 'pose_y', 'goal', 'robot', 'environment', 'env_type','augmentation'],


limo_clearpath_playpen_nodes_reference_df = pd.read_csv(f"../dataframes/closest_node_analysis/limo_clearpath_playpen_nodes_reference.csv") # node_idx, node_x_odom, node_y_odom

reference_nodes = {
("limo", "clearpath_playpen"): limo_clearpath_playpen_nodes_reference_df,
    # add more if needed
}


all_data_with_errors = compute_node_errors(df, reference_nodes)
all_data_with_errors.columns

In [ ]:
all_data = pd.merge(all_data_with_errors, metrics_df, on=['robot', 'environment', 'augmentation'], how='left')
all_data.columns

## single plot

In [ ]:
test_df = all_data[all_data["augmentation"] == "rain_torrential"]
test_df.columns

# plot_single_radar_chart(all_data)



In [ ]:
test_df[['node_error', 'mean_jerk', 'path_length', 'path_efficiency', 'avg_velocity', 'avg_curvature']].head()

In [ ]:
test_df["mean_jerk"].max()

In [ ]:
test_np = np.array([1,2,0,4,5,0])
test_mask = test_np != 0
test_idx = np.where(test_mask)[0]
test_idx

In [ ]:
df_overtake = all_data_with_errors[all_data_with_errors["augmentation"] == "rain"]
df_overtake["lin_x_teleop"].any()

# TESTING


In [ ]:
def compute_velocity(segment_df, dt):
    """
    Compute average velocity for a path segment.
    
    Args:
        segment_df: DataFrame with pose_x and pose_y columns
        dt: Time step between consecutive poses
    
    Returns:
        Average velocity in units/second
    """
    if len(segment_df) < 2:
        return 0.0
    
    # Calculate distance between consecutive points
    distances = []
    for i in range(len(segment_df) - 1):
        x1, y1 = segment_df.iloc[i]['pose_x'], segment_df.iloc[i]['pose_y']
        x2, y2 = segment_df.iloc[i + 1]['pose_x'], segment_df.iloc[i + 1]['pose_y']
        dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        distances.append(dist)
    
    # Average velocity = total distance / total time
    total_distance = sum(distances)
    total_time = len(distances) * dt
    avg_velocity = total_distance / total_time if total_time > 0 else 0.0
    
    return avg_velocity


def compute_path_length(segment_df):
    """
    Compute total path length for a trajectory.
    
    Args:
        segment_df: DataFrame with pose_x and pose_y columns
    
    Returns:
        Total path length
    """
    if len(segment_df) < 2:
        return 0.0
    
    total_length = 0.0
    for i in range(len(segment_df) - 1):
        x1, y1 = segment_df.iloc[i]['pose_x'], segment_df.iloc[i]['pose_y']
        x2, y2 = segment_df.iloc[i + 1]['pose_x'], segment_df.iloc[i + 1]['pose_y']
        dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        total_length += dist
    
    return total_length


def compute_cumulative_heading_change(segment_df):
    """
    Compute cumulative heading changes (measure of smoothness).
    Lower values = smoother path.
    
    Args:
        segment_df: DataFrame with pose_x and pose_y columns
    
    Returns:
        Total heading change in radians
    """
    if len(segment_df) < 3:
        return 0.0
    
    total_heading_change = 0.0
    
    for i in range(len(segment_df) - 2):
        # Get three consecutive points
        x1, y1 = segment_df.iloc[i]['pose_x'], segment_df.iloc[i]['pose_y']
        x2, y2 = segment_df.iloc[i + 1]['pose_x'], segment_df.iloc[i + 1]['pose_y']
        x3, y3 = segment_df.iloc[i + 2]['pose_x'], segment_df.iloc[i + 2]['pose_y']
        
        # Calculate vectors
        vec1_x, vec1_y = x2 - x1, y2 - y1
        vec2_x, vec2_y = x3 - x2, y3 - y2
        
        # Calculate angles
        angle1 = np.arctan2(vec1_y, vec1_x)
        angle2 = np.arctan2(vec2_y, vec2_x)
        
        # Calculate heading change (normalized to [-pi, pi])
        heading_change = angle2 - angle1
        heading_change = np.arctan2(np.sin(heading_change), np.cos(heading_change))
        
        total_heading_change += abs(heading_change)
    
    return total_heading_change


def compute_path_similarity(path1_df, path2_df):
    """
    Compute similarity between two paths using Dynamic Time Warping.
    Returns a similarity score (higher = more similar).
    
    Args:
        path1_df: First path DataFrame with pose_x and pose_y
        path2_df: Second path (reference) DataFrame with pose_x and pose_y
    
    Returns:
        DTW distance (lower = more similar)
    """
    if len(path1_df) < 2 or len(path2_df) < 2:
        return float('inf')
    
    # Extract coordinates
    path1 = path1_df[['pose_x', 'pose_y']].values
    path2 = path2_df[['pose_x', 'pose_y']].values
    
    # Compute DTW distance
    distance, _ = fastdtw(path1, path2, dist=euclidean)
    
    return distance


def normalize_metrics(metrics_df, metric_columns):
    """
    Normalize metrics to [0, 1] range.
    For velocity and path_similarity: higher is better (1 = best)
    For path_length and heading_change: lower is better (1 = best)
    
    Args:
        metrics_df: DataFrame with computed metrics
        metric_columns: Dictionary specifying which metrics to normalize
                       {'column_name': 'higher_is_better' or 'lower_is_better'}
    
    Returns:
        DataFrame with normalized columns added
    """
    normalized_df = metrics_df.copy()
    
    for col, direction in metric_columns.items():
        if col not in normalized_df.columns:
            continue
        
        min_val = normalized_df[col].min()
        max_val = normalized_df[col].max()
        
        if max_val == min_val:
            # All values are the same
            normalized_df[f'{col}_normalized'] = 1.0
        else:
            # Normalize to [0, 1]
            normalized = (normalized_df[col] - min_val) / (max_val - min_val)
            
            # If lower is better, invert the scale
            if direction == 'lower_is_better':
                normalized = 1.0 - normalized
            
            normalized_df[f'{col}_normalized'] = normalized
    
    return normalized_df


def compute_all_metrics(all_data, dt):
    """
    Compute all metrics for each combination of robot, augmentation, and environment.
    
    Args:
        all_data: Full DataFrame
        dt: Time step between poses
        model_only_segments: Function to extract model-only segments
    
    Returns:
        DataFrame with computed metrics for each group
    """
    results = []
    
    # Get reference paths for each robot-environment combination
    reference_paths = {}
    
    # Group by robot, environment, augmentation
    groups = all_data.groupby(['robot', 'environment', 'augmentation'])
    
    for (robot, env, aug), group_df in groups:
        # Get model-only segments
        segments = model_only_segments(group_df)
        
        if len(segments) == 0:
            print(f"No model-only segments for ({robot}, {env}, {aug})")
            continue
        
        # Process each segment
        for start_idx, end_idx in segments:
            segment_df = group_df.iloc[start_idx:end_idx + 1].copy()
            
            if len(segment_df) < 2:
                continue
            
            # Compute metrics
            velocity = compute_velocity(segment_df, dt)
            path_length = compute_path_length(segment_df)
            heading_change = compute_cumulative_heading_change(segment_df)
            
            # Store reference path
            if aug == 'reference':
                ref_key = (robot, env)
                if ref_key not in reference_paths:
                    reference_paths[ref_key] = []
                reference_paths[ref_key].append(segment_df)
            
            results.append({
                'robot': robot,
                'environment': env,
                'augmentation': aug,
                'segment_start': start_idx,
                'segment_end': end_idx,
                'avg_velocity': velocity,
                'path_length': path_length,
                'cumulative_heading_change': heading_change,
                'segment_df': segment_df
            })
    
    # Compute path similarities
    for result in results:
        robot = result['robot']
        env = result['environment']
        aug = result['augmentation']
        ref_key = (robot, env)
        
        if aug != 'reference' and ref_key in reference_paths:
            # Compare with reference path(s)
            similarities = []
            for ref_path in reference_paths[ref_key]:
                sim = compute_path_similarity(result['segment_df'], ref_path)
                similarities.append(sim)
            result['path_similarity_dtw'] = min(similarities) if similarities else float('inf')
        else:
            result['path_similarity_dtw'] = 0.0
    
    # Convert to DataFrame
    metrics_df = pd.DataFrame(results)
    metrics_df = metrics_df.drop('segment_df', axis=1)
    
    # Normalize metrics
    metric_columns = {
        'avg_velocity': 'higher_is_better',
        'path_length': 'lower_is_better',
        'cumulative_heading_change': 'lower_is_better',
        'path_similarity_dtw': 'lower_is_better'
    }
    
    metrics_df = normalize_metrics(metrics_df, metric_columns)
    
    return metrics_df


# Example usage:
# dt = 0.1  # time step in seconds
# metrics = compute_all_metrics(all_data, dt, model_only_segments)
# print(metrics[['robot', 'environment', 'augmentation', 
#                'avg_velocity_normalized', 'path_length_normalized',
#                'cumulative_heading_change_normalized', 'path_similarity_dtw_normalized']])

In [ ]:
all_data = all_data_with_errors.copy()
all_data.columns

In [ ]:
all_data["augmentation"].unique()

In [ ]:
# Compute all metrics
all_data.fillna(0.0, inplace=True)
metrics = compute_all_metrics(all_data, DT)



In [ ]:
# View results
print(metrics[['robot', 'environment', 'augmentation', 
               'avg_velocity_normalized', 'path_length_normalized',
               'cumulative_heading_change_normalized', 
               'path_similarity_dtw_normalized']])

In [ ]:
type(metrics)

In [ ]:
metrics["augmentation"].unique()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import colorsys

def adjust_color_brightness(hex_color, factor=1.2):
    """Lighten or darken a hex color."""
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    h, l, s = colorsys.rgb_to_hls(r/255, g/255, b/255)
    l = max(0, min(1, l * factor))
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'



def plot_single_radar_chart(
    df, 
    metrics=['Vel', 'Len', 'Δθ', 'Sim'], 
    robot_type='limo', 
    environment_type='clearpath_playpen',
    color_sequence=None,
    is_first_figure=True
):
    """
    Plot a polar (radar) chart where each polygon corresponds to a data augmentation type.
    Each axis corresponds to a normalized metric value.
    """
    
    filtered_df = df[
        (df['robot'] == robot_type) &
        (df['environment'] == environment_type)
    ]
    
    augmentations = filtered_df['augmentation'].unique()
    fig = go.Figure()

    # Choose a color palette if not provided
    if color_sequence is None:
        color_sequence = px.colors.qualitative.Plotly


    for i, augmentation_type in enumerate(augmentations):
        aug_df = filtered_df[filtered_df['augmentation'] == augmentation_type]

        metric_values = [aug_df[m].mean() for m in metrics]
        plot_df = pd.DataFrame({
            'metric': metrics,
            'value': metric_values
        })

        # Close the polygon
        plot_df = pd.concat([plot_df, plot_df.iloc[[0]]])

        if isinstance(color_sequence, dict):
            base_color = color_sequence.get(augmentation_type, '#888888')
        else:
            base_color = color_sequence[i % len(color_sequence)]


        line_color = adjust_color_brightness(base_color, 0.7)
        fill_color = adjust_color_brightness(base_color, 1.4)


        fig.add_trace(go.Scatterpolar(
            r=plot_df['value'],
            theta=plot_df['metric'],
            fill='toself',
            name=augmentation_type,
            line=dict(color=line_color, width=2),
            fillcolor=fill_color,
            opacity=0.5
        ))


    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                range=[0, 1],
                showticklabels=True,
                ticks=''
            )
        ),
        showlegend=True,
    )

    if is_first_figure:

        fig.update_layout(
            legend=dict(
                title="Augmentations",     # optional
                orientation="v",          # 'h' for horizontal, 'v' for vertical
                yanchor="middle",         # anchor position
                y=0.5,                   # vertical position
                xanchor="left",
                x=0.2                     # horizontal position (0 = left, 1 = right)
            )
     )

    fig.show(renderer="notebook")
    return fig


In [ ]:
color_sequence = {
"blur": "#3A5CF0", 
"brightness": "#EB8E75", 
"fog": "#93EF98",
"no_augmentation": "#F3C282",
"rain": "#82F1F5", 
"rain_drizzle":  "#11BFFE",
"rain_heavy":  "#7EF9AF",
"rain_torrential":  '#06D6A0',
"reference":  "#FF90F9",
"sunFlare_overlay":  "#EFB6FF",
"sunFlare_physic":  "#F74269"
}

In [ ]:
polar_chart_all_metrics = metrics.copy()
polar_chart_all_metrics.rename(columns={
                                'cumulative_heading_change_normalized': 'Δθ',
                                'path_similarity_dtw_normalized': 'Sim',
                                'path_length_normalized': 'Len',
                                'avg_velocity_normalized': 'Vel'
                            }, inplace=True)
polar_chart_all_metrics.columns

In [ ]:

polar_chart_all = plot_single_radar_chart(polar_chart_all_metrics, color_sequence=color_sequence)

**RAIN ONLY HERE**

In [ ]:

rain_only_metrics = polar_chart_all_metrics[
    polar_chart_all_metrics["augmentation"].str.contains("rain|rain_drizzle|rain_torrential|rain_heavy|reference", case=False, na=False)
]
rain_only_metrics["augmentation"].unique()

polar_chart_rain = plot_single_radar_chart(rain_only_metrics, color_sequence=color_sequence)

In [ ]:

# TODO NOT WORKING 
# Use of plotly.io.kaleido.scope.default_format is deprecated and support will be removed after September 2025.
# Please use plotly.io.defaults.default_format instead
#  pio.kaleido.scope.default_format = "pdf"
# BrowserFailedError: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`. It may be your copy auto-updated.')

# polar_chart_rain.write_image("../medias/polar_metrics_nomad.pdf")

# TRAJ PLOTS 3D

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

def plot_3d_trajectories(df, robot_type, environment_type, palette=None, is_first_figure=True):
    # Filter dataset
    filtered_df = df[
        (df["robot"] == robot_type) &
        (df["environment"] == environment_type)
    ].copy()
    
    # Add timestep per trajectory
    filtered_df["timestep"] = filtered_df.groupby(
        ["robot", "environment", "augmentation"]
    ).cumcount()

    # Default color palette
    if palette is None:
        palette = {
        "blur": "#3A5CF0", 
        "brightness": "#EB8E75", 
        "fog": "#93EF98",
        "no_augmentation": "#F3C282",
        "rain": "#82F1F5", 
        "rain_drizzle":  "#11BFFE",
        "rain_heavy":  "#7EF9AF",
        "rain_torrential":  '#06D6A0',
        "reference":  "#FF90F9",
        "sunFlare_overlay":  "#EFB6FF",
        "sunFlare_physic":  "#F74269"
        }

    # Define visually distinct line properties
    opacities = [1.0, 0.9, 0.8, 0.75, 0.7]

    fig = go.Figure()

    for i, (aug_name, group) in enumerate(filtered_df.groupby("augmentation")):
        color = palette.get(aug_name, "#999999")
        opacity = opacities[i % len(opacities)]

        # Add main trajectory line
        fig.add_trace(go.Scatter3d(
            x=group["pose_x"],
            y=group["pose_y"],
            z=group["timestep"],
            mode="lines+markers",
            name=aug_name,
            line=dict(width=3.0, color=color),
            marker=dict(size=2, color=color, opacity=0),  # invisible intermediate markers
            opacity=opacity,
        ))

        # Add endpoint marker (same color)
        end_point = group.iloc[-1]
        fig.add_trace(go.Scatter3d(
            x=[end_point["pose_x"]],
            y=[end_point["pose_y"]],
            z=[end_point["timestep"]],
            mode="markers",
            marker=dict(size=6, color=color, symbol="circle"),
            showlegend=False  # don’t duplicate legend
        ))

        if aug_name == "reference":
            fig.add_trace(go.Scatter3d(
                x=[group["pose_x"].iloc[0]], y=[group["pose_y"].iloc[0]], z=[0],
                mode='markers', marker=dict(size=4.5, color='green', symbol="diamond"), name=f"Start"
            ))


    fig.add_trace(go.Scatter3d(
        x=[None], y=[None], z=[None],   
        mode="markers",
        marker=dict(size=5, color="black", symbol="circle"),
        name="end (colored by augmentation)"
    ))

    # Layout
    fig.update_layout(
        scene=dict(
            xaxis_title="X position [m]",
            yaxis_title="Y position [m]",
            zaxis_title="Timestep",
            aspectmode="cube"
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    if is_first_figure:
        fig.update_layout(
            legend=dict(
                title="Augmentations",     # optional
                orientation="v",          # 'h' for horizontal, 'v' for vertical
                yanchor="middle",         # anchor position
                y=0.5,                   # vertical position
                xanchor="left",
                x=0.2                     # horizontal position (0 = left, 1 = right)
            )
     )

    fig.show()
    return fig


In [ ]:
_ = plot_3d_trajectories(
    df,
    robot_type="limo",
    environment_type="clearpath_playpen"
)


# DIST ERROR PLOT

In [ ]:
df_node_errors = all_data_with_errors.copy()
df_node_errors.columns

In [ ]:
df_node_errors = df_node_errors.sort_values(by=["robot", "environment", "augmentation"]).copy()
df_node_errors["timestep"] = df_node_errors.groupby(["robot", "environment", "augmentation"]).cumcount()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def plot_node_error(df, robot, environment, title=None, save_path=None, show=True, palette=None, reference_header='reference'):
    # Filter to a specific robot/environment
    subset = df[(df["robot"] == robot) & (df["environment"] == environment) & (df["augmentation"] != reference_header)]

    # Default color palette (custom tint)
    if palette is None:
        palette = {
            "blur": "#3A5CF0", 
            "brightness": "#EB8E75", 
            "fog": "#93EF98",
            "no_augmentation": "#F3C282",
            "rain": "#82F1F5", 
            "rain_drizzle":  "#11BFFE",
            "rain_heavy":  "#7EF9AF",
            "rain_torrential":  '#06D6A0',
            "reference":  "#FF90F9",
            "sunFlare_overlay":  "#EFB6FF",
            "sunFlare_physic":  "#F74269"
        }

    sns.set(style="darkgrid", context="talk")
    plt.figure(figsize=(8, 6))

    # Use your palette directly here
    sns.lineplot(
        data=subset,
        x="timestep",
        y="node_error",
        hue="augmentation",
        linewidth=2.5,
        palette=palette   # ✅ use your color dictionary
    )

    plt.xlabel("Discrete Timestep")
    plt.ylabel("Predicted Node Error")
    plt.title(title or f"Node Error over Time – {robot}, {environment}")

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close()


In [ ]:
df_node_errors["node_error"].max()
df_node_errors["node_error"].min()


In [ ]:
df_node_error_downsampled = df_node_errors[df_node_errors["timestep"] % 100 == 0]

plot_node_error(df_node_error_downsampled, robot="limo", environment="clearpath_playpen")